In [1]:
!pip install pytorch-crf -q

In [2]:
import pandas as pd
import numpy as np
import torch
import random

from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer
)

In [3]:
SEED = 42

EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

MODEL_NAME = "csebuetnlp/banglabert"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)
print("Model:", MODEL_NAME)

Device: cuda
Model: csebuetnlp/banglabert


In [4]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Seed:", SEED)

Seed: 42


In [5]:
datasets = {
    "Barishal": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Barishal_NER.csv"),
    "Chittagong": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Chittagong_NER.csv"),
    "Mymensingh": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Mymensingh_NER.csv"),
    "Noakhali": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Noakhali_NER.csv"),
    "Sylhet": pd.read_csv("/kaggle/input/datasets/arnav1605/ancholink-ner/Sylhet_NER.csv")
}

print("Datasets loaded.")

Datasets loaded.


In [6]:
sentence_data = {}

for dialect, df in datasets.items():

    df["Sentence #"] = df["Sentence #"].ffill()

    word_column = [
        c for c in df.columns
        if c.endswith("_word")
    ][0]

    sentences = []

    for sentence_id, group in df.groupby("Sentence #"):

        sentences.append({
            "sentence_id": int(
                str(sentence_id).replace("Sentence: ", "")
            ),
            "tokens": group[word_column].astype(str).tolist(),
            "tags": group["bio_tag"].astype(str).tolist()
        })

    sentence_data[dialect] = sentences

for dialect in sentence_data:
    print(
        dialect,
        "→",
        len(sentence_data[dialect]),
        "sentences"
    )

Barishal → 3481 sentences
Chittagong → 3481 sentences
Mymensingh → 3481 sentences
Noakhali → 3481 sentences
Sylhet → 3481 sentences


In [7]:
tags = set()

for sentences in sentence_data.values():

    for sentence in sentences:
        tags.update(sentence["tags"])

tags = sorted(tags)

tag_to_id = {
    tag: i
    for i, tag in enumerate(tags)
}

id_to_tag = {
    i: tag
    for tag, i in tag_to_id.items()
}

print("Number of tags:", len(tags))
print(tag_to_id)

Number of tags: 20
{'\nI-LOC': 0, 'B-ANI': 1, 'B-COL': 2, 'B-FOOD': 3, 'B-LOC': 4, 'B-OBJ': 5, 'B-ORG': 6, 'B-PER': 7, 'B-REL': 8, 'B-ROLE': 9, 'I-ANI': 10, 'I-COL': 11, 'I-FOOD': 12, 'I-LOC': 13, 'I-OBJ': 14, 'I-ORG': 15, 'I-PER': 16, 'I-REL': 17, 'I-ROLE': 18, 'O': 19}


In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded.")

config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded.


In [9]:
def get_cross_dialect_data(held_out):

    train_sentences = []
    test_sentences = sentence_data[held_out]

    for dialect in sentence_data:

        if dialect != held_out:
            train_sentences.extend(
                sentence_data[dialect]
            )

    return train_sentences, test_sentences

In [10]:
def make_dataset(sentences):

    return Dataset.from_dict({
        "tokens": [
            s["tokens"]
            for s in sentences
        ],
        "tags": [
            s["tags"]
            for s in sentences
        ]
    })

In [11]:
def tokenize_and_align_labels(example):

    tokenized = tokenizer(
        example["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=128
    )

    word_ids = tokenized.word_ids()

    labels = []

    previous_word_id = None

    for word_id in word_ids:

        if word_id is None:
            labels.append(-100)

        else:
            labels.append(
                tag_to_id[
                    example["tags"][word_id]
                ]
            )

        previous_word_id = word_id

    tokenized["labels"] = labels

    return tokenized

In [12]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(tag_to_id),
    id2label=id_to_tag,
    label2id=tag_to_id
)

model.to(device)

print("Model loaded.")

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded.


In [16]:
def get_entities(labels):
    entities = []
    start = None
    entity_type = None

    for i, tag in enumerate(labels + ["O"]):

        if tag.startswith("B-"):
            if start is not None:
                entities.append((start, i - 1, entity_type))

            start = i
            entity_type = tag[2:]

        elif tag.startswith("I-"):
            if start is None:
                start = i
                entity_type = tag[2:]

        else:
            if start is not None:
                entities.append((start, i - 1, entity_type))

            start = None
            entity_type = None

    return set(entities)

In [13]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    correct = 0
    predicted = 0
    actual = 0

    for pred_row, label_row in zip(
        predictions,
        labels
    ):

        true_tags = []
        pred_tags = []

        for pred, true in zip(
            pred_row,
            label_row
        ):

            if true == -100:
                continue

            true_tags.append(
                id_to_tag[true]
            )

            pred_tags.append(
                id_to_tag[pred]
            )

        true_entities = get_entities(
            true_tags
        )

        pred_entities = get_entities(
            pred_tags
        )

        correct += len(
            true_entities & pred_entities
        )

        actual += len(true_entities)
        predicted += len(pred_entities)

    precision = (
        correct / predicted
        if predicted > 0 else 0
    )

    recall = (
        correct / actual
        if actual > 0 else 0
    )

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if precision + recall > 0 else 0
    )

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [14]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    output_dir="./banglabert_cross_dialect",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=20,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="no",
    report_to="none",
    seed=42
)

In [17]:
banglabert_results = []

for held_out in sentence_data:

    print("\n" + "=" * 60)
    print("Held-out Dialect:", held_out)
    print("=" * 60)

    # Create train and test data
    train_sentences, test_sentences = get_cross_dialect_data(
        held_out
    )

    # Convert to Hugging Face datasets
    train_dataset = make_dataset(train_sentences)
    test_dataset = make_dataset(test_sentences)

    # Tokenize and align labels
    train_tokenized = train_dataset.map(
        tokenize_and_align_labels
    )

    test_tokenized = test_dataset.map(
        tokenize_and_align_labels
    )

    # Create a NEW BanglaBERT model for this fold
    model = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(tag_to_id),
        id2label=id_to_tag,
        label2id=tag_to_id
    )

    # Create Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tokenized,
        eval_dataset=test_tokenized,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    # Train
    trainer.train()

    # Evaluate
    metrics = trainer.evaluate()

    banglabert_results.append({
        "Held-out Dialect": held_out,
        "Precision": metrics["eval_precision"],
        "Recall": metrics["eval_recall"],
        "F1": metrics["eval_f1"]
    })

    print("\nResults:")
    print(
        f"Precision: {metrics['eval_precision']:.4f}"
    )
    print(
        f"Recall:    {metrics['eval_recall']:.4f}"
    )
    print(
        f"F1:        {metrics['eval_f1']:.4f}"
    )


Held-out Dialect: Barishal


Map:   0%|          | 0/13924 [00:00<?, ? examples/s]

Map:   0%|          | 0/3481 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

ElectraForTokenClassification LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     | 
--------------------------------------------------+------------+-
electra.embeddings.position_ids                   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.weight                                 | MISSING    | 
classifier.bias                                   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
100,1.389332
200,0.813408


KeyboardInterrupt: 

In [ ]:
banglabert_results = pd.DataFrame(
    banglabert_results
)

banglabert_results

In [ ]:
banglabert_results.to_csv(
    "/kaggle/working/banglabert_cross_dialect_results.csv",
    index=False
)

banglabert_summary.to_csv(
    "/kaggle/working/banglabert_cross_dialect_summary.csv"
)

print("BanglaBERT cross-dialect results saved.")